In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine, text


# database connections 


In [ ]:
with open('../../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['OLTP']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)


# Extract


In [ ]:
sede = pd.read_sql_query("""
SELECT s.sede_id, s.nombre AS sede_nombre, s.direccion AS sede_direccion,
       s.telefono AS sede_telefono, s.cliente_id, s.ciudad_id
FROM sede s
""", co_sa)
cliente = pd.read_sql_query("""
SELECT c.cliente_id, c.nombre AS nombre_cliente, c.nit_cliente,
       c.sector AS sector_cliente, tc.nombre AS tipo_cliente
FROM cliente c
LEFT JOIN tipo_cliente tc ON tc.tipo_cliente_id = c.tipo_cliente_id
""", co_sa)
ciudad = pd.read_sql_query("""
SELECT ci.ciudad_id, ci.nombre AS ciudad, ci.departamento_id,
       d.nombre AS departamento
FROM ciudad ci
LEFT JOIN departamento d ON d.departamento_id = ci.departamento_id
""", co_sa)
sede.head()


# Transformations


In [ ]:
dim_ubicacion = sede.merge(cliente, on='cliente_id', how='left')
dim_ubicacion = dim_ubicacion.merge(ciudad, on='ciudad_id', how='left')
dim_ubicacion.rename(columns={
    'sede_id': 'id_sede',
    'cliente_id': 'id_cliente',
    'ciudad_id': 'id_ciudad',
    'departamento_id': 'id_departamento',
}, inplace=True)
dim_ubicacion.replace({np.nan: 'NO APLICA', ' ': 'NO APLICA', '': 'NO APLICA'}, inplace=True)
for col in ['id_sede', 'id_cliente', 'id_ciudad', 'id_departamento']:
    dim_ubicacion[col] = pd.to_numeric(dim_ubicacion[col], errors='coerce').fillna(-1).astype('Int64')

unknown = pd.DataFrame([{
    'id_sede': -1, 'sede_nombre': 'NO APLICA', 'sede_direccion': 'NO APLICA', 'sede_telefono': 'NO APLICA',
    'id_cliente': -1, 'nombre_cliente': 'NO APLICA', 'nit_cliente': 'NO APLICA',
    'sector_cliente': 'NO APLICA', 'tipo_cliente': 'NO APLICA',
    'id_ciudad': -1, 'ciudad': 'NO APLICA', 'id_departamento': -1, 'departamento': 'NO APLICA',
}])
dim_ubicacion = pd.concat([unknown, dim_ubicacion], ignore_index=True)
dim_ubicacion["saved"] = date.today()
dim_ubicacion.head()


# load


In [ ]:
with etl_conn.begin() as conn:
    conn.execute(text('Delete from dim_ubicacion'))
dim_ubicacion.to_sql('dim_ubicacion', etl_conn, if_exists='append', index=False)
